# SmartCart Day 5b - Gradio Demo

The finale: a **self-checkout demo**. Upload a counter photo, the app detects products, recognizes each via the ONNX head, finds catalog matches, and totals the basket from a simple price table - the whole week's work behind one button.

In [ ]:
# 1) Runtime setup
# This course runs in Google Colab. Run this cell first.
# Install only the packages used in this notebook.
%pip install -q timm ultralytics onnxruntime gradio

import os

# Drive stores the small cross-day bundle, not the image dataset.
from google.colab import drive
drive.mount('/content/drive')
BUNDLE_DIR = '/content/drive/MyDrive/SmartCart'



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 51.2 MB/s eta 0:00:00
Mounted at /content/drive


### Embedded toolkit

This cell defines the helper functions used below. Run it once after setup.

In [ ]:
from __future__ import annotations
import json
import os
import pathlib
import subprocess
import numpy as np
import pandas as pd
HERE = pathlib.Path.cwd()  # embedded in-notebook: no __file__, anchor on the working dir
GROCERY_DATASET_URL = 'https://github.com/marcusklasson/GroceryStoreDataset'

class Bundle:
    """Small Google Drive folder that carries artifacts from one day to the next."""

    def __init__(self, root: str):
        self.root = pathlib.Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.manifest = {'version': 1, 'class_list': [], 'artifacts': {}}

    def put_table(self, name, df: pd.DataFrame):
        df.to_csv(self.root / name, index=False)
        self._note(name)

    def get_table(self, name) -> pd.DataFrame:
        return pd.read_csv(self.root / name)

    def put_array(self, name, arr: np.ndarray):
        np.save(self.root / name, arr)
        self._note(name)

    def get_array(self, name) -> np.ndarray:
        p = self.root / name
        return np.load(p if p.suffix == '.npy' else p.with_suffix('.npy'))

    def _note(self, name):
        self.manifest['artifacts'][name] = True

    def save(self):
        (self.root / 'manifest.json').write_text(json.dumps(self.manifest, indent=2))

    def load(self):
        p = self.root / 'manifest.json'
        if p.exists():
            self.manifest = json.loads(p.read_text())
        return self

def open_bundle(drive_dir='/content/drive/MyDrive/SmartCart') -> Bundle:
    """Open the cross-day Drive bundle. If it is new, start with an empty manifest."""
    return Bundle(drive_dir).load()

def save_bundle(b: Bundle):
    b.save()
    print(f'[bundle] saved -> {b.root}')

def load_backbone(name='vit_small_patch14_dinov2.lvd142m', device='cpu'):
    """Frozen DINOv2-small feature extractor."""
    import timm
    m = timm.create_model(name, pretrained=True, num_classes=0, dynamic_img_size=True).eval().to(device)
    for p in m.parameters():
        p.requires_grad_(False)
    return m

def extract_features(model, batches, device=None) -> np.ndarray:
    """Run image batches through a frozen backbone and return numpy features."""
    import torch
    if device is None:
        try:
            device = next(model.parameters()).device
        except (AttributeError, StopIteration):
            device = 'cpu'
    outs = []
    with torch.no_grad():
        for xb in batches:
            y = model(xb.to(device))
            outs.append(y.detach().cpu().numpy().astype('float32'))
    return np.concatenate(outs, 0)

class EmbeddingIndex:
    """Cosine nearest-neighbor index over catalog/gallery embeddings."""

    def __init__(self):
        self._nn = None
        self.meta = []

    def build(self, feats: np.ndarray, meta: list[dict]):
        from sklearn.neighbors import NearestNeighbors
        f = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-08)
        self._nn = NearestNeighbors(n_neighbors=min(10, len(f)), metric='cosine').fit(f)
        self.meta = meta
        self._feats = f
        return self

    def query(self, q: np.ndarray, k=5):
        qn = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-08)
        dist, idx = self._nn.kneighbors(qn, n_neighbors=min(k, len(self.meta)))
        return [[{**self.meta[j], 'score': float(1 - d)} for d, j in zip(drow, irow)] for drow, irow in zip(dist, idx)]

    def save(self, b: Bundle, prefix='gallery'):
        b.put_array(f'{prefix}_index.npy', self._feats)
        b.put_table(f'{prefix}_meta.csv', pd.DataFrame(self.meta))

    @classmethod
    def load(cls, b: Bundle, prefix='gallery'):
        feats = b.get_array(f'{prefix}_index.npy')
        meta = b.get_table(f'{prefix}_meta.csv').to_dict('records')
        return cls().build(feats, meta)

def basket_total(items, prices, default=0.0):
    return float(sum((prices.get(it['fine'], default) for it in items)))

def score_onnx_logits(logits, classes):
    """Same softmax-confidence logic as Day 3/5a's score_image()/score_feat(), applied
    to raw ONNX Runtime output instead of a PyTorch head - this notebook serves via
    sess.run(), not head.net(), so the scoring math has to be redone against that shape."""
    logits = logits[0] if logits.ndim == 2 else logits
    probs = np.exp(logits - logits.max())
    probs = probs / probs.sum()
    pred_id = int(np.argmax(logits))
    return {'pred_id': pred_id, 'pred_name': classes[pred_id], 'confidence': float(probs[pred_id])}

# Confidence below this is flagged as needing manual review in the live demo, rather
# than trusted silently. Tune with Day 3's evaluate_confidence_thresholds() sweep;
# 0.5 is a placeholder, not a validated cutoff (same caveat as Day 5a).
CONFIDENCE_THRESHOLD = 0.5
# --- helpers are now available as plain functions/classes in this notebook ---
print('SmartCart toolkit ready')


SmartCart toolkit ready


In [ ]:
# 2) Load the cross-day bundle
# The bundle stores artifacts we create during the week: labels, indexes, weights, ONNX files.
# It is NOT the image dataset. Images are loaded in the next data-source cell.
b = open_bundle(BUNDLE_DIR)
print('bundle:', b.root)
print('artifacts:', list(b.manifest.get('artifacts', {})))


bundle: /content/drive/MyDrive/SmartCart
artifacts: ['gallery_index.npy', 'gallery_meta.csv', 'labels.csv', 'catalog_prices.csv', 'detector.pt', 'crops_manifest.csv', 'head.pt', 'per_class_metrics.csv', 'error_report.md', 'head_v2.pt', 'lift_table.csv', 'sample_scene.jpg', 'head.onnx']


## Load model + catalog prices

**What:** Open an ONNX session for the head, load the gallery, load `catalog_prices.csv`, and load the detector + backbone.

**Why:** These are the exact runtime pieces the app needs - no training, just serving.

**Watch for:** Prices are editable demo business data; the ONNX head replaces PyTorch at serve time.

In [ ]:
from pathlib import Path
import onnxruntime as ort
import pandas as pd
import torch
# Load serving-time artifacts: ONNX head, gallery, price catalog, backbone, detector.
sess = ort.InferenceSession(str(Path(b.root)/'head.onnx'))
idx = EmbeddingIndex.load(b)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device:', device)
model = load_backbone(device=device)
classes = b.manifest.get('class_list') or sorted({m['fine'] for m in idx.meta})
price_catalog = b.get_table('catalog_prices.csv')
prices = dict(zip(price_catalog.fine, price_catalog.unit_price))
currency = price_catalog.currency.iloc[0] if 'currency' in price_catalog else 'USD'
from ultralytics import YOLO
det = YOLO(str(Path(b.root)/'detector.pt'))
print('serving', len(classes), 'classes with', len(prices), 'catalog prices in', currency)


using device: cuda


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
serving 81 classes with 81 catalog prices in USD


## Inference function

**What:** Define `recognize(image)`: detect -> crop -> embed -> ONNX argmax -> label, then total the basket.

**Why:** This single function is the app's brain; Gradio just wraps it with a UI.

**Watch for:** Returns an annotated image, a per-item table, and the basket total.

In [ ]:
import torch, torchvision.transforms.v2 as T
from PIL import Image
import numpy as np
# Shared image preprocessing for DINOv2.
TF = T.Compose([T.ToImage(), T.Resize((224,224)), T.ToDtype(torch.float32, scale=True),
                T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
def feats_of(paths, model, bs=16):
    """Embed image files in small batches and return an (N, D) numpy array."""
    out=[]
    for i in range(0,len(paths),bs):
        xs=torch.stack([TF(Image.open(p).convert('RGB')) for p in paths[i:i+bs]])
        out.append(extract_features(model,[xs]))
    return np.concatenate(out,0)
from PIL import Image, ImageDraw
def feats_of_imgs(imgs, model):
    """Embed PIL crops without saving them to disk."""
    xs = torch.stack([TF(im.convert('RGB')) for im in imgs])
    return extract_features(model, [xs])
def recognize(image):
    img = image.convert('RGB'); arr = np.array(img); draw = ImageDraw.Draw(img)
    result = det(arr, imgsz=640, conf=0.05, verbose=False)[0]
    boxes = result.boxes.xyxy.cpu().numpy()
    if len(boxes) == 0:
        boxes = np.array([[0, 0, img.width, img.height]], dtype='float32')
    items, rows = [], []
    # One row per detected product. Confidence is checked here, not just predicted -
    # this is the live demo end users actually click, so this is where "unknown item /
    # confidence warning" (from the project brief) has to actually take effect.
    for x1,y1,x2,y2 in boxes:
        crop = Image.fromarray(arr[int(y1):int(y2), int(x1):int(x2)])
        f = feats_of_imgs([crop], model)
        logits = sess.run(None, {'feat': f.astype('float32')})[0]
        s = score_onnx_logits(logits, classes)
        label = s['pred_name']
        needs_review = s['confidence'] < CONFIDENCE_THRESHOLD
        similar = [h['fine'] for h in idx.query(f, k=3)[0]]
        items.append({'fine': label})
        rows.append({'item': label, 'confidence': round(s['confidence'], 3), 'needs_review': needs_review,
                     'price': prices.get(label, 0.0), 'currency': currency, 'similar': similar})
        box_label = f"{label} ({s['confidence']:.0%})" + (' ?' if needs_review else '')
        box_color = 'orange' if needs_review else 'red'
        draw.rectangle([x1,y1,x2,y2], outline=box_color, width=3); draw.text((x1,y1), box_label, fill=box_color)
    total = basket_total(items, prices, default=1.0)  # unknown SKUs priced at a flat $1.00
    n_review = sum(r['needs_review'] for r in rows)
    return img, {'items': rows, 'total': round(total, 2), 'currency': currency,
                 'needs_review_count': n_review, 'note': 'includes best-guess price for flagged items' if n_review else None}
print('recognize() ready')


recognize() ready


## Launch the app

**What:** Launch the Gradio interface (guarded so CI doesn't block on a server).

**Why:** A shareable link lets students try the self-checkout from their phones.

**Watch for:** Set SC_LAUNCH_GRADIO=1 in Colab to launch; CI just prints a hint.

In [ ]:
os.environ['SC_LAUNCH_GRADIO'] = '1'
if os.environ.get('SC_LAUNCH_GRADIO','0')=='1':
    import gradio as gr
    demo = gr.Interface(fn=recognize, inputs=gr.Image(type='pil'), outputs=['image','json'], title='SmartCart self-checkout')
    demo.launch(share=True)
else:
    print('Set SC_LAUNCH_GRADIO=1 (in Colab) to launch the shareable app.')


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed750c8879da3c0f3b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Showcase + leaderboard

Drop your best counter photo into the app and screenshot the basket. Post your validation accuracy (Day 3) and rare-class lift (Day 4) to the class leaderboard — we celebrate the biggest honest lift, not just the highest single number.

## Week recap / where next

You built the full deployable-vision loop: **acquire -> localize -> recognize -> diagnose -> augment -> assemble -> ship**. From here, see `deep_dive/` for the optional generative-augmentation stack, true open-set recognition, and on-device quantization-aware training. Same bundle contract, deeper each step.